In [1]:
import os
import torch
import torchvision.models as models
from torch.hub import download_url_to_file

def download_resnet18_weights(save_path='/root/autodl-tmp/resnet_weights/resnet18-5c106cde.pth'):
    """
    下载ResNet18预训练权重文件
    """
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    if not os.path.exists(save_path):
        print(f"正在下载ResNet18预训练权重到 {save_path}...")
        try:
            # 方法1: 使用torchvision自带的下载功能
            model = models.resnet18(pretrained=True)  # 改为resnet18
            torch.save(model.state_dict(), save_path)
            print("✅ 通过torchvision下载成功")
        except Exception as e:
            print(f"torchvision下载失败: {str(e)}")
            print("尝试直接下载权重文件...")
            try:
                # 方法2: 直接从PyTorch Hub下载
                url = "https://download.pytorch.org/models/resnet18-5c106cde.pth"  # 改为ResNet18的URL
                download_url_to_file(url, save_path)
                print("✅ 直接下载成功")
            except Exception as e:
                print(f"直接下载失败: {str(e)}")
                raise RuntimeError("无法下载ResNet18预训练权重")
    else:
        print(f"✅ 权重文件已存在: {save_path}")
    
    return save_path

In [1]:
import os
import shutil
import random

# 原始路径
image_src_dir = "/root/autodl-tmp/data/train/images"
mask_src_dir = "/root/autodl-tmp/data/train/masks"

# 目标路径
image_train_dst = "/root/autodl-tmp/data_1/train/images"
mask_train_dst = "/root/autodl-tmp/data_1/train/masks"
image_val_dst = "/root/autodl-tmp/data_1/val/images"
mask_val_dst = "/root/autodl-tmp/data_1/val/masks"

# 创建目标目录
os.makedirs(image_train_dst, exist_ok=True)
os.makedirs(mask_train_dst, exist_ok=True)
os.makedirs(image_val_dst, exist_ok=True)
os.makedirs(mask_val_dst, exist_ok=True)

# 获取所有图像文件路径并建立映射关系
file_pairs = []
for folder in os.listdir(image_src_dir):
    if not folder.isdigit():
        continue
        
    folder_path = os.path.join(image_src_dir, folder)
    for img_file in os.listdir(folder_path):
        if not img_file.endswith('.png'):
            continue
            
        img_num = img_file.split('.')[0]
        img_path = os.path.join(folder_path, img_file)
        mask_path = os.path.join(mask_src_dir, folder, img_file)
        
        if os.path.exists(mask_path):
            new_name = f"{folder}_{img_num}.png"
            file_pairs.append((img_path, mask_path, new_name))

# 随机打乱文件对
random.shuffle(file_pairs)

# 划分训练集和验证集 (999:1)
split_idx = int(len(file_pairs) * 0.999)
train_pairs = file_pairs[:split_idx]
val_pairs = file_pairs[split_idx:]

# 处理训练集
for img_path, mask_path, new_name in train_pairs:
    # 复制并重命名图像
    shutil.copy2(img_path, os.path.join(image_train_dst, new_name))
    # 复制并重命名掩码
    shutil.copy2(mask_path, os.path.join(mask_train_dst, new_name))

# 处理验证集
for img_path, mask_path, new_name in val_pairs:
    # 复制并重命名图像
    shutil.copy2(img_path, os.path.join(image_val_dst, new_name))
    # 复制并重命名掩码
    shutil.copy2(mask_path, os.path.join(mask_val_dst, new_name))

print(f"处理完成！")
print(f"训练集图像数量: {len(train_pairs)}")
print(f"验证集图像数量: {len(val_pairs)}")
print(f"验证集占比: {len(val_pairs)/len(file_pairs)*100:.2f}%")

OSError: [Errno 28] No space left on device: '/root/autodl-tmp/data_1/train/masks/89_478.png'